# CSIRO Image2Biomass: Dual-Stream DINO Inference & Submission

This notebook runs **Offline & Online Ensemble Inference** for the CSIRO Image2Biomass competition using models trained locally or in Kaggle.

### 🏆 Architecture Highlights
1. **Dual-Stream 1:1 Square Splitting**: $2000 \times 1000 \to$ Left ($1000 \times 1000$) & Right ($1000 \times 1000$) views resized to $512 \times 512$.
2. **Cross-View Multi-Head Self-Attention**: Tokens interact across the panoramic pasture seam.
3. **Automatic Checkpoint Discovery**: Finds and ensembles all uploaded `.pt` or `.pth` model checkpoints.
4. **Test-Time Augmentation (TTA)**: Evaluates normal views and horizontally flipped mirrored views.
5. **Soft Physical Post-Processing**: Blends $GDM = 0.5 \cdot GDM + 0.5 \cdot (Green + Clover)$ and $Total = 0.5 \cdot Total + 0.5 \cdot (Green + Clover + Dead)$.

### 🚀 How to Use:
1. **Upload your trained models**:
   - Create a Kaggle Dataset (e.g. `csiro-dino-models`) with your `.pt` files (`best_model_fold1.pt`, `best_model_fold2.pt`, etc.).
2. **Attach to this notebook**:
   - Click **+ Add Input** on the right panel $\to$ Add your model dataset.
   - Ensure the competition dataset (`csiro-biomass`) is attached.
3. **Run All**:
   - The notebook automatically finds your models, runs inference, and generates `submission.csv` ready for scoring.


In [ ]:
# 1. Environment Setup & Offline Support
import os
import sys
import glob
import random
import subprocess
import numpy as np
import pandas as pd
from PIL import Image
import cv2
from tqdm.auto import tqdm

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms

# Handle timm: works online or from offline .whl if attached
try:
    import timm
except ImportError:
    wheels = glob.glob('/kaggle/input/**/timm*.whl', recursive=True)
    if len(wheels) > 0:
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', wheels[0]])
    else:
        subprocess.check_call(['pip', 'install', '-q', 'timm'])
    import timm

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {DEVICE}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')


In [ ]:
# 2. Configuration & Automatic Path & Model Discovery
def find_data_dir():
    candidates = [
        '/kaggle/input/competitions/csiro-biomass',
        '/kaggle/input/csiro-biomass',
        '../input/competitions/csiro-biomass',
        '../input/csiro-biomass',
        './data',
        '.'
    ]
    for c in candidates:
        if os.path.exists(os.path.join(c, 'test.csv')):
            return c
    if os.path.exists('/kaggle/input'):
        for root, _, files in os.walk('/kaggle/input'):
            if 'test.csv' in files:
                return root
    return '.'

def find_model_checkpoints():
    """Finds all uploaded model checkpoint (.pt or .pth) files across /kaggle/input and working directory."""
    found = []
    if os.path.exists('/kaggle/input'):
        for root, _, files in os.walk('/kaggle/input'):
            # Skip image directories
            if any(x in root.lower() for x in ['train', 'test', 'images', 'csiro-biomass/train']):
                continue
            for f in files:
                if f.endswith('.pt') or f.endswith('.pth'):
                    found.append(os.path.join(root, f))
    
    for f in sorted(glob.glob('*.pt') + glob.glob('*.pth')):
        p = os.path.abspath(f)
        if p not in found:
            found.append(p)
            
    model_files = [f for f in found if 'model' in f.lower() or 'fold' in f.lower()]
    return sorted(model_files if model_files else found)

class CFG:
    DATA_DIR = find_data_dir()
    TEST_CSV = os.path.join(DATA_DIR, 'test.csv')
    SAMPLE_SUB_CSV = os.path.join(DATA_DIR, 'sample_submission.csv')
    TEST_IMG_DIR = os.path.join(DATA_DIR, 'test') if os.path.exists(os.path.join(DATA_DIR, 'test')) else DATA_DIR
    
    BACKBONE = 'vit_base_patch16_dinov3_qkvb'
    IMG_SIZE = 512
    FUSION_DIM = 384
    NUM_INTERVALS = 7
    BATCH_SIZE = 8
    USE_TTA = True
    
    TARGET_ORDER = ['Dry_Green_g', 'Dry_Dead_g', 'Dry_Clover_g', 'GDM_g', 'Dry_Total_g']
    IMAGENET_MEAN = [0.485, 0.456, 0.406]
    IMAGENET_STD = [0.229, 0.224, 0.225]

print(f'DATA_DIR: {CFG.DATA_DIR}')
print(f'TEST_CSV: {CFG.TEST_CSV} (Exists: {os.path.exists(CFG.TEST_CSV)})')

CHECKPOINTS = find_model_checkpoints()
print(f'\nDiscovered {len(CHECKPOINTS)} Model Checkpoint(s):')
for cp in CHECKPOINTS:
    print(f'  - {cp} ({os.path.getsize(cp) / (1024*1024):.1f} MB)')

if len(CHECKPOINTS) == 0:
    print('\n[!] Note: No .pt/.pth checkpoints found yet.')
    print('    Please upload your trained models as a Kaggle dataset and attach it via "+ Add Input".')


In [ ]:
# 3. Dual-Stream Architecture (pretrained=False for offline execution)
class DualStreamBiomassModel(nn.Module):
    def __init__(self, backbone_name=CFG.BACKBONE, num_targets=5, num_intervals=7, fusion_dim=384, dropout=0.3, pretrained=False, **kwargs):
        super().__init__()
        # pretrained=False: Loads structure cleanly without network downloads
        self.backbone = timm.create_model(backbone_name, pretrained=pretrained, num_classes=0)
        self.backbone_dim = self.backbone.num_features
        
        num_heads = 8 if self.backbone_dim % 8 == 0 else 4
        self.cross_view_attn = nn.MultiheadAttention(embed_dim=self.backbone_dim, num_heads=num_heads, dropout=0.1, batch_first=True)
        self.attn_norm = nn.LayerNorm(self.backbone_dim)
        
        self.fusion_mlp = nn.Sequential(
            nn.Linear(self.backbone_dim * 2, fusion_dim),
            nn.LayerNorm(fusion_dim),
            nn.GELU(),
            nn.Dropout(dropout)
        )
        
        self.reg_heads = nn.ModuleList([
            nn.Sequential(
                nn.Linear(fusion_dim, fusion_dim // 2),
                nn.LayerNorm(fusion_dim // 2),
                nn.GELU(),
                nn.Dropout(dropout * 0.5),
                nn.Linear(fusion_dim // 2, 64),
                nn.GELU(),
                nn.Linear(64, 1)
            ) for _ in range(num_targets)
        ])
        
        self.cls_heads = nn.ModuleList([
            nn.Sequential(
                nn.Linear(fusion_dim, 128),
                nn.LayerNorm(128),
                nn.GELU(),
                nn.Dropout(dropout * 0.5),
                nn.Linear(128, num_intervals)
            ) for _ in range(num_targets)
        ])

    def extract_features(self, x):
        feats = self.backbone(x)
        return feats.mean(dim=1) if len(feats.shape) == 3 else feats.mean(dim=[2, 3]) if len(feats.shape) == 4 else feats

    def forward(self, img_left, img_right):
        feat_l = self.extract_features(img_left)
        feat_r = self.extract_features(img_right)
        
        tokens = torch.stack([feat_l, feat_r], dim=1)
        attn_out, _ = self.cross_view_attn(tokens, tokens, tokens)
        tokens = self.attn_norm(tokens + attn_out)
        
        fused = self.fusion_mlp(torch.cat([tokens[:, 0], tokens[:, 1]], dim=-1))
        reg_preds = [F.softplus(head(fused)) for head in self.reg_heads]
        cls_preds = [head(fused) for head in self.cls_heads]
        return reg_preds, cls_preds


In [ ]:
# 4. Dual-Stream Test Dataset & Image Resolver
class DualStreamTestDataset(Dataset):
    def __init__(self, df, img_dir=CFG.TEST_IMG_DIR, img_size=CFG.IMG_SIZE):
        self.df = df.reset_index(drop=True)
        self.img_dir = img_dir
        self.img_size = img_size
        self.transform = transforms.Compose([
            transforms.Resize((img_size, img_size)),
            transforms.ToTensor(),
            transforms.Normalize(mean=CFG.IMAGENET_MEAN, std=CFG.IMAGENET_STD),
        ])

    def __len__(self):
        return len(self.df)

    def _resolve_image_path(self, raw_path):
        if os.path.exists(raw_path):
            return raw_path
        fname = os.path.basename(raw_path)
        candidates = [
            os.path.join(CFG.DATA_DIR, raw_path),
            os.path.join(CFG.DATA_DIR, 'test', fname),
            os.path.join(CFG.DATA_DIR, 'train', fname),
            os.path.join(self.img_dir, fname) if self.img_dir else None,
            os.path.join(self.img_dir, raw_path) if self.img_dir else None,
            os.path.join('/kaggle/input/competitions/csiro-biomass', raw_path),
            os.path.join('/kaggle/input/csiro-biomass', raw_path),
            os.path.join('test', fname),
            os.path.join('train', fname)
        ]
        for c in candidates:
            if c and os.path.exists(c):
                return c
        for root, _, files in os.walk(CFG.DATA_DIR):
            if fname in files:
                return os.path.join(root, fname)
        raise FileNotFoundError(f'Image {fname} not found in {CFG.DATA_DIR}')

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        img_path = self._resolve_image_path(row['image_path'])
        raw_bgr = cv2.imread(img_path)
        if raw_bgr is None:
            raise ValueError(f'Failed to load image from: {img_path}')
        raw_rgb = cv2.cvtColor(raw_bgr, cv2.COLOR_BGR2RGB)
        
        mid_w = raw_rgb.shape[1] // 2
        left_np = raw_rgb[:, :mid_w].copy()
        right_np = raw_rgb[:, mid_w:].copy()
        
        tensor_l = self.transform(Image.fromarray(left_np))
        tensor_r = self.transform(Image.fromarray(right_np))
        
        return {
            'image_left': tensor_l,
            'image_right': tensor_r,
            'clean_id': row.get('clean_id', row.get('sample_id', f'sample_{idx}')),
        }


In [ ]:
# 5. Soft Physical Post-Processing & Calibration
def soft_physics_postprocess(preds_np):
    """Enforces physical relationships and non-negativity across targets:"""
    preds = np.maximum(preds_np.copy(), 0.0)
    green = preds[:, 0]
    dead = preds[:, 1]
    clover = preds[:, 2] * 0.8
    gdm = preds[:, 3]
    total = preds[:, 4]
    
    dead = np.where(dead > 20.0, dead * 1.1, np.where(dead < 10.0, dead * 0.9, dead))
    gdm_blended = 0.5 * gdm + 0.5 * (green + clover)
    total_blended = 0.5 * total + 0.5 * (green + clover + dead)
    
    return np.maximum(np.column_stack([green, dead, clover, gdm_blended, total_blended]), 0.0)


In [ ]:
# 6. Multi-Checkpoint Ensemble Inference with TTA
assert len(CHECKPOINTS) > 0, 'ERROR: No checkpoints found! Please attach your trained models dataset to the notebook.'

# 1. Read Test Data
test_df_raw = pd.read_csv(CFG.TEST_CSV)
if 'target_name' in test_df_raw.columns:
    test_df_raw['clean_id'] = test_df_raw['sample_id'].astype(str).apply(lambda x: x.split('__')[0])
    unique_test = test_df_raw[['clean_id', 'image_path']].drop_duplicates(subset=['clean_id']).reset_index(drop=True)
else:
    unique_test = test_df_raw.copy()
    if 'clean_id' not in unique_test.columns:
        unique_test['clean_id'] = unique_test['sample_id']

print(f'Test set: {len(unique_test)} unique pasture images to predict.')

# 2. Create DataLoader
test_dataset = DualStreamTestDataset(unique_test, CFG.TEST_IMG_DIR, CFG.IMG_SIZE)
test_loader = DataLoader(test_dataset, batch_size=CFG.BATCH_SIZE, shuffle=False, num_workers=2)

# 3. Predict across all discovered model checkpoints
all_fold_predictions = []

for cp_idx, cp_path in enumerate(CHECKPOINTS):
    print(f'\n[{cp_idx + 1}/{len(CHECKPOINTS)}] Evaluating: {os.path.basename(cp_path)}...')
    model = DualStreamBiomassModel(CFG.BACKBONE, pretrained=False).to(DEVICE)
    
    state_dict = torch.load(cp_path, map_location=DEVICE)
    state_dict = {k.replace('module.', ''): v for k, v in state_dict.items()}
    model.load_state_dict(state_dict)
    model.eval()
    
    fold_preds = []
    with torch.no_grad():
        for batch in tqdm(test_loader, desc=f'Inference {os.path.basename(cp_path)}', leave=False):
            img_l = batch['image_left'].to(DEVICE)
            img_r = batch['image_right'].to(DEVICE)
            
            if CFG.USE_TTA:
                # Standard view
                r_std, _ = model(img_l, img_r)
                # Mirrored panoramic view (flip right view as left, flip left view as right)
                r_flip, _ = model(torch.flip(img_r, [3]), torch.flip(img_l, [3]))
                avg_r = [(a + b) * 0.5 for a, b in zip(r_std, r_flip)]
            else:
                avg_r, _ = model(img_l, img_r)
                
            pred_batch = torch.cat(avg_r, dim=1).cpu().numpy()
            fold_preds.append(pred_batch)
            
    all_fold_predictions.append(np.concatenate(fold_preds, axis=0))

# 4. Ensemble Average & Soft Physical Post-Processing
ensemble_raw = np.mean(all_fold_predictions, axis=0)
ensemble_post = soft_physics_postprocess(ensemble_raw)
print(f'\n✓ Successfully ensembled predictions from {len(CHECKPOINTS)} model checkpoint(s)!')


In [ ]:
# 7. Generate and Verify submission.csv
clean_ids = [s['clean_id'] for s in test_dataset]
pred_dict = {
    clean_ids[i]: {col: float(ensemble_post[i, c_idx]) for c_idx, col in enumerate(CFG.TARGET_ORDER)}
    for i in range(len(clean_ids))
}

if 'target_name' in test_df_raw.columns:
    submission_df = test_df_raw.copy()
    submission_df['target'] = submission_df.apply(
        lambda row: pred_dict.get(row['clean_id'], {}).get(row['target_name'], 0.0),
        axis=1
    )
    final_sub = submission_df[['sample_id', 'target']].copy()
elif os.path.exists(CFG.SAMPLE_SUB_CSV):
    sub_template = pd.read_csv(CFG.SAMPLE_SUB_CSV)
    sub_template['clean_id'] = sub_template['sample_id'].astype(str).apply(lambda x: x.split('__')[0])
    sub_template['target_name'] = sub_template['sample_id'].astype(str).apply(lambda x: x.split('__')[1])
    sub_template['target'] = sub_template.apply(
        lambda row: pred_dict.get(row['clean_id'], {}).get(row['target_name'], 0.0),
        axis=1
    )
    final_sub = sub_template[['sample_id', 'target']].copy()
else:
    records = []
    for cid in clean_ids:
        for col in CFG.TARGET_ORDER:
            records.append({
                'sample_id': f'{cid}__{col}',
                'target': pred_dict[cid][col]
            })
    final_sub = pd.DataFrame(records)

# Quality and Integrity checks
assert final_sub['target'].isna().sum() == 0, 'ERROR: Submission contains NaN values!'
assert len(final_sub) > 0, 'ERROR: Submission dataframe is empty!'

output_path = 'submission.csv'
final_sub.to_csv(output_path, index=False)

print('=' * 50)
print(f'SUCCESS! Saved submission to: {output_path}')
print(f'Total predictions: {len(final_sub)}')
print('=' * 50)
print('\nTarget Distribution:')
print(final_sub['target'].describe())
print('\nFirst 10 Rows:')
print(final_sub.head(10))
